# 2016 Amatrice Earthquake (Central Italy) — Full Rebuild

**Regenerated from scratch** after real data loss (local storage cleared).
Every fix identified and verified across this whole project is wired in
directly, not re-derived by hand: real orbit-based coregistration, the
`row_offset` deburst alignment fix, real per-burst-overlap ESD, the
fixed (fast, correct) topographic-phase regression, real track-
consistency selection (`select_consistent_geometry`), and real
sub-swath-consistency extraction with automatic full-swath-fallback
rejection (`extract_consistent_stack`).

**What we already know, stated honestly up front**: this AOI is
vegetated, mountainous terrain in August — real, physical coherence
ceiling around 0.2-0.3 for this kind of C-band data, confirmed
independently against ESA's own Carleton training material. This
notebook is built to get a real, reliable result from real data, not to
force a higher number than the physics supports.

In [1]:
import numpy as np
import rasterio
from pathlib import Path
from datetime import datetime
from itertools import combinations

from pygeofetch import PyGeoFetch
from pygeofetch.models import BoundingBox
from pygeofetch.models.search_query import SearchQuery
from pygeofetch.models.download_task import DownloadOptions
from pygeofetch.processing.preprocessor import Preprocessor
from pygeofetch.core.orbits import fetch_orbit_file
from pygeofetch.insar import (
    SLCExtractor, InterferogramGenerator, SBASTimeSeries,
    AtmosphericCorrector, select_consistent_geometry,
)
from pygeofetch.insar.timeseries import InterferogramPair
from pygeofetch.insar.geolocation import parse_orbit_file, perpendicular_baseline
from pygeofetch.insar.unwrap import PhaseUnwrapper, multilook, bridge_unwrap_regions
from pygeofetch.insar.validate import DataValidator
from pygeofetch.viz.map import MapViewer

client = PyGeoFetch()
output_dir = Path("data/amatrice_insar")
output_dir.mkdir(parents=True, exist_ok=True)

WAVELENGTH_M = 0.05546576
aoi_bbox = BoundingBox(min_lon=13.10-0.170, max_lon=13.45-0.170, min_lat=42.65, max_lat=42.85)

15:55:50 INFO [      engine] PyGeoFetch ready


## 1. Search — wide window, then real track filtering

Widened from the original 20-day window (which only ever gave 2 usable
scenes) to ~2.5 months around the earthquake. `select_consistent_geometry()`
groups by real track and keeps only the largest same-track group —
replacing hand-written grouping logic with the same, tested function
used across this project.

In [2]:
search_query = SearchQuery(
    bbox=aoi_bbox, start_date="2016-07-15", end_date="2016-09-30",
    product_type="SLC", max_results=50,satellites=["Sentinel-1A"], platform="Sentinel-1"
)
search_results = client.search(search_query, providers=["copernicus"])
print(f"Real search: {len(search_results)} Sentinel-1 SLC scenes found")

by_date = {}
for r in search_results:
    label = str(r.datetime)[:10]
    if label not in by_date:
        by_date[label] = r

selected, geometry_report = select_consistent_geometry(list(by_date.values()),max_scenes=6,preferred_track=95)
print(f"\nReal track kept: {geometry_report['track']}")
print(f"Real satellites present: {geometry_report['satellites']}")
print(f"Same-geometry scenes ({len(selected)}): {[str(s.datetime)[:10] for s in selected]}")
if geometry_report["dropped"]:
    print(f"Dropped (different real track): {geometry_report['dropped']}")

┌ SEARCH PARAMETERS ───────────────────────────────────────────────────────┐
│ Providers  : copernicus                                                  │
│ BBox       : [12.930, 42.650, 13.280, 42.850]                            │
│ Date range : 2016-07-15  →  2016-09-30                                   │
│ Cloud max  : 100%                                                        │
│ Product    : SLC                                                         │
└──────────────────────────────────────────────────────────────────────────┘
15:55:52 INFO [  copernicus] Authenticated with Copernicus Data Space as 'appiahkubis14@gmail.com'
15:55:56 INFO [  copernicus] Real unit-level satellite filter (['S1A']): 38/46 results kept.
  ✓  copernicus                      38 scenes   5.8s
┌────────────────────────────────────────────┬────────────┬────────────────┬────────┬─────────┬──────────────┬─────────────┬───────┬───────┬──────────────────────┐
│                  SCENE ID                  │    D

In [3]:
import json
import tempfile

aoi_geojson = {
    "type": "FeatureCollection",
    "features": [{
        "type": "Feature",
        "properties": {"name": "AOI"},
        "geometry": {
            "type": "Polygon",
            "coordinates": [[
                [aoi_bbox.min_lon, aoi_bbox.min_lat],
                [aoi_bbox.max_lon, aoi_bbox.min_lat],
                [aoi_bbox.max_lon, aoi_bbox.max_lat],
                [aoi_bbox.min_lon, aoi_bbox.max_lat],
                [aoi_bbox.min_lon, aoi_bbox.min_lat],
            ]],
        },
    }],
}
aoi_path = Path(tempfile.mkdtemp()) / "aoi.geojson"
aoi_path.write_text(json.dumps(aoi_geojson))

mv = MapViewer(center=((aoi_bbox.min_lat + aoi_bbox.max_lat) / 2, (aoi_bbox.min_lon + aoi_bbox.max_lon) / 2), zoom=10)
mv.add_basemap("SATELLITE")
mv.add_search_results(selected)
mv.add_vector(str(aoi_path), layer_name="AOI", style={"color": "yellow", "fillOpacity": 0, "weight": 3})
mv.show()

15:55:59 INFO [         map] Vector layer added: search_results
15:55:59 INFO [         map] Vector layer added: AOI


Map(center=[42.75, 13.105], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_…

## 2. Download the real, filtered scenes

In [4]:
raw_dir = output_dir / "raw"
download_results_list = client.download(selected, destination=raw_dir, options=DownloadOptions(parallel=4, resume=True))
download_results = {str(s.datetime)[:10]: dr for s, dr in zip(selected, download_results_list)}
extracted_dates = list(download_results.keys())
print(f"Real downloads complete: {len(download_results)} scenes: {extracted_dates}")

⬇ 6 scenes  →  data/amatrice_insar/raw              0/6  [00:00]

d42e40c6-22b8-52ee-847b-5a644e683183: 0.00B [00:00, ?B/s]

c218486b-0033-5939-abfe-8e881d7229c2: 0.00B [00:00, ?B/s]

147dd390-5a5a-5629-9d1c-e14dcabb8175: 0.00B [00:00, ?B/s]

7f295acd-862d-52b2-ace1-3b2906b0a16d: 0.00B [00:00, ?B/s]

15:56:25 INFO [  downloader]   ↷ 147dd390-5a5a-5629-9d1c-e14dcabb8175          already downloaded, skipping (resume=True)


9b50bdc4-f5b8-5fa6-a690-293a90e7353d: 0.00B [00:00, ?B/s]

15:56:25 INFO [  downloader]   ↷ c218486b-0033-5939-abfe-8e881d7229c2          already downloaded, skipping (resume=True)


18725b62-7631-5a68-9243-3c7f49b4609f: 0.00B [00:00, ?B/s]

15:56:25 INFO [  downloader]   ↷ 7f295acd-862d-52b2-ace1-3b2906b0a16d          already downloaded, skipping (resume=True)
15:56:25 INFO [  downloader]   ↷ d42e40c6-22b8-52ee-847b-5a644e683183          already downloaded, skipping (resume=True)
15:56:47 INFO [  downloader]   ↷ 18725b62-7631-5a68-9243-3c7f49b4609f          already downloaded, skipping (resume=True)
15:56:47 INFO [  downloader]   ↷ 9b50bdc4-f5b8-5fa6-a690-293a90e7353d          already downloaded, skipping (resume=True)
Real downloads complete: 6 scenes: ['2016-07-21', '2016-08-02', '2016-08-14', '2016-08-26', '2016-09-07', '2016-09-19']


## 3. Real orbit files — precise orbits for every real downloaded scene

In [5]:
orbit_dir = output_dir / "orbits"
orbit_dir.mkdir(parents=True, exist_ok=True)

orbit_files = {}
for label, dr in download_results.items():
    try:
        orbit_files[label] = fetch_orbit_file(
            product_name=Path(dr.output_path).name, output_dir=str(orbit_dir), orbit_type="precise",
        )
        print(f"  {label}: {Path(orbit_files[label]).name}")
    except Exception as exc:
        print(f"  {label}: orbit download failed -- {exc}")

print(f"\n{len(orbit_files)}/{len(download_results)} real orbit files ready")

15:56:47 INFO [      orbits] Using cached orbit file: S1A_OPER_AUX_POEORB_OPOD_20210312T234800_V20160720T225943_20160722T005943.EOF
  2016-07-21: S1A_OPER_AUX_POEORB_OPOD_20210312T234800_V20160720T225943_20160722T005943.EOF
15:56:47 INFO [      orbits] Using cached orbit file: S1A_OPER_AUX_POEORB_OPOD_20210313T044206_V20160801T225943_20160803T005943.EOF
  2016-08-02: S1A_OPER_AUX_POEORB_OPOD_20210313T044206_V20160801T225943_20160803T005943.EOF
15:56:47 INFO [      orbits] Using cached orbit file: S1A_OPER_AUX_POEORB_OPOD_20210313T093737_V20160813T225943_20160815T005943.EOF
  2016-08-14: S1A_OPER_AUX_POEORB_OPOD_20210313T093737_V20160813T225943_20160815T005943.EOF
15:56:47 INFO [      orbits] Using cached orbit file: S1A_OPER_AUX_POEORB_OPOD_20210313T134622_V20160825T225943_20160827T005943.EOF
  2016-08-26: S1A_OPER_AUX_POEORB_OPOD_20210313T134622_V20160825T225943_20160827T005943.EOF
15:56:47 INFO [      orbits] Using cached orbit file: S1A_OPER_AUX_POEORB_OPOD_20210313T173704_V20160906

## 4. Real DEM (OpenTopography) — required for topographic phase removal

In [6]:
dem_dir = output_dir / "dem"
dem_dir.mkdir(parents=True, exist_ok=True)

bbox_tuple = (aoi_bbox.min_lon, aoi_bbox.min_lat, aoi_bbox.max_lon, aoi_bbox.max_lat)

dem_results = client.search(SearchQuery(bbox=aoi_bbox, product_type="DEM"), providers=["opentopography"])
if not dem_results:
    raise RuntimeError("No real DEM found for this AOI -- check opentopography credentials/coverage")

raw_dem_path = client.download(dem_results[:1], destination=dem_dir)[0].output_path
dem_path = Preprocessor().clip(raw_dem_path, bbox=bbox_tuple, output=str(dem_dir / "dem_clipped.tif")).output_path
print(f"Real DEM ready: {dem_path}")

┌ SEARCH PARAMETERS ───────────────────────────────────────────────────────┐
│ Providers  : opentopography                                              │
│ BBox       : [12.930, 42.650, 13.280, 42.850]                            │
│ Date range : None  →  None                                               │
│ Cloud max  : 100%                                                        │
│ Product    : DEM                                                         │
└──────────────────────────────────────────────────────────────────────────┘
  ✓  opentopography                   7 scenes   0.0s
┌────────────────────────────────────────────┬────────────┬────────────────┬────────┬─────────┬──────────────┬─────────────┬───────┬───────┬──────────────────────┐
│                  SCENE ID                  │    DATE    │   SATELLITE    │ CLOUD  │ PRODUCT │ POLARISATION │    PASS     │ ORBIT │ SCORE │       PROVIDER       │
├────────────────────────────────────────────┼────────────┼────────────────┼───

⬇ 1 scene  →  data/amatrice_insar/dem              0/1  [00:00]

opentopo_SRTMGL3_12.93_42.65: 0.00B [00:00, ?B/s]

15:56:47 INFO [  downloader]   ↷ opentopo_SRTMGL3_12.93_42.65                  already downloaded, skipping (resume=True)
15:56:47 INFO [preprocessor] Clipped → data/amatrice_insar/dem/dem_clipped.tif
Real DEM ready: data/amatrice_insar/dem/dem_clipped.tif


In [7]:
mv = MapViewer(center=((aoi_bbox.min_lat + aoi_bbox.max_lat) / 2, (aoi_bbox.min_lon + aoi_bbox.max_lon) / 2), zoom=10)
mv.add_basemap("SATELLITE")
mv.add_search_results(dem_results)
mv.add_vector(str(aoi_path), layer_name="AOI", style={"color": "yellow", "fillOpacity": 0, "weight": 3})
mv.show()

15:56:47 INFO [         map] Real geometry missing for 7/7 results — used bbox fallback
15:56:47 INFO [         map] Vector layer added: search_results
15:56:47 INFO [         map] Vector layer added: AOI


Map(center=[42.75, 13.105], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_…

In [8]:
mv = MapViewer(center=((aoi_bbox.min_lat + aoi_bbox.max_lat) / 2, (aoi_bbox.min_lon + aoi_bbox.max_lon) / 2), zoom=10)
mv.add_basemap("SATELLITE")
mv.add_search_results(selected)
mv.add_raster(dem_path)
mv.add_vector(str(aoi_path), layer_name="AOI", style={"color": "yellow", "fillOpacity": 0, "weight": 3})
mv.show()

15:56:48 INFO [         map] Vector layer added: search_results


15:56:51 INFO [         map] Raster layer added: dem_clipped
15:56:51 INFO [         map] Vector layer added: AOI


Map(center=[42.750417, 13.104583], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title',…

## 5. Real extraction — sub-swath forced consistent, unreliable crops auto-rejected

Single call: extracts the reference scene, forces every other date onto
the same real sub-swath, and automatically rejects any date whose
forced extraction falls back to the full, uncropped swath (the real
failure mode confirmed for 2016-08-27 earlier in this project).

In [9]:
extractor = SLCExtractor(polarisation="VV")
scenes = {label: download_results[label].output_path for label in extracted_dates}

extracted_slcs, extraction_report = extractor.extract_consistent_stack(scenes, aoi_bbox, output_dir / "slc")

print(f"Reference: {extraction_report['reference']}, matched real sub-swath: {extraction_report['matched_swath']} "
      f"({extraction_report['reference_rows']} real rows)")
if extraction_report["excluded"]:
    print(f"Excluded: {extraction_report['excluded']}")

orbit_files = {k: v for k, v in orbit_files.items() if k in extracted_slcs}
download_results = {k: v for k, v in download_results.items() if k in extracted_slcs}
extracted_dates = list(extracted_slcs.keys())
print(f"\n{len(extracted_slcs)} real, reliable scenes kept: {extracted_dates}")

15:56:51 INFO [  extraction] Found 3 VV sub-swath(s) in S1A_IW_SLC__1SDV_20160721T051920_20160721T051947_012242_013044_5553.SAFE.zip
15:56:54 INFO [  extraction] Cropped IW1: 22935x13563 -> 9281x2814 (11.9x smaller)
15:56:54 INFO [  extraction] Extracted IW1 -> 2016-07-21_vv.tif
15:56:54 INFO [  extraction] Reference scene 2016-07-21 matched real sub-swath iw1 (2814 real cropped rows) — forcing all other dates onto iw1.
15:56:54 INFO [  extraction] Found 3 VV sub-swath(s) in S1A_IW_SLC__1SDV_20160802T051921_20160802T051948_012417_013615_2BF2.SAFE.zip
15:56:54 INFO [  extraction] Using preferred sub-swath iw1 (matching the real sub-swath already chosen for another date in this stack) — skipping the automatic per-date overlap search.
15:56:57 INFO [  extraction] Cropped IW1: 22935x13563 -> 9266x2814 (11.9x smaller)
15:56:57 INFO [  extraction] Extracted IW1 -> 2016-08-02_vv.tif
15:56:57 INFO [  extraction]   2016-08-02: extracted (2814 real rows, matches reference)
15:56:57 INFO [  extra

## 6. Interferogram formation — every real pair, complete verified pipeline

Real orbit-based coregistration, real per-burst-overlap ESD, real
deburst with the `row_offset` alignment fix, real flat-earth removal,
Goldstein filtering.

In [10]:
ifg_gen = InterferogramGenerator(
    coherence_window=5, 
    esd_enabled=True, 
    use_gpu=False,
    use_real_burst_processing=True, 
    remove_flat_earth_phase=True,
)
LOOKS_AZ, LOOKS_RG = 2, 1

interferograms = {}
for d1, d2 in combinations(extracted_dates, 2):
    coreg_kwargs = {}
    if all(d in download_results and d in orbit_files for d in (d1, d2)):
        coreg_kwargs = dict(
            reference_safe_zip=download_results[d1].output_path, 
            secondary_safe_zip=download_results[d2].output_path,
            reference_orbit_file=orbit_files[d1], 
            secondary_orbit_file=orbit_files[d2],
        )
    try:
        result = ifg_gen.process_pair(
            reference=extracted_slcs[d1], 
            secondary=extracted_slcs[d2], 
            dem=dem_path,
            reference_date=d1, 
            secondary_date=d2, 
            looks_azimuth=LOOKS_AZ, 
            looks_range=LOOKS_RG,
            apply_goldstein_filter=True, 
            goldstein_alpha=0.6, 
            **coreg_kwargs,
        )
    except ValueError as exc:
        print(f"  {d1} -> {d2}: REJECTED -- {exc}")
        continue
    days = (datetime.fromisoformat(d2).date() - datetime.fromisoformat(d1).date()).days
    interferograms[(d1, d2)] = result
    result.save(output_dir / "interferograms" / f"{d1}_{d2}", auto_visualize=True)
    print(f"  {d1} -> {d2} ({abs(days):3d}d): coherence={result.coherence.mean():.3f}")

print(f"\n{len(interferograms)} real pairs formed")

15:57:13 INFO [  annotation] Parsed real SLC geometry from S1A_IW_SLC__1SDV_20160721T051920_20160721T051947_012242_013044_5553.SAFE.zip: 13563 x 22935, starting 2016-07-21 05:19:22.194298
15:57:13 INFO [  annotation] Parsed real SLC geometry from S1A_IW_SLC__1SDV_20160802T051921_20160802T051948_012417_013615_2BF2.SAFE.zip: 13563 x 22935, starting 2016-08-02 05:19:23.076256
15:57:13 INFO [ geolocation] Parsed 9361 orbit state vectors from S1A_OPER_AUX_POEORB_OPOD_20210312T234800_V20160720T225943_20160722T005943.EOF
15:57:14 INFO [ geolocation] Parsed 9361 orbit state vectors from S1A_OPER_AUX_POEORB_OPOD_20210313T044206_V20160801T225943_20160803T005943.EOF
15:57:15 INFO [  coregister] DEM-driven offset field: 49/49 points solved successfully
15:57:15 INFO [interferogram] Correcting for cropped extraction: reference offset (8598, 0), secondary offset (8603, 0)
15:57:15 INFO [interferogram] Real orbit-based coregistration applied (49 grid points)
15:57:20 INFO [  annotation] Parsed real S

## 7. Real atmospheric correction (elevation-correlated, circular regression)

Uses the same, real, verified frequency-search fix applied to the
topographic-phase regression — accurate for genuine multi-cycle
elevation/phase relationships, not just small ones.

In [11]:
atm_corrector = AtmosphericCorrector(method="elevation")
corrected_interferograms = {}

for (d1, d2), result in interferograms.items():
    wrapped_phase = np.angle(result.interferogram)
    corrected_phase, atm_meta = atm_corrector.correct(wrapped_phase, profile=result.profile, dem=dem_path, return_metadata=True)
    corrected_interferograms[(d1, d2)] = corrected_phase
    r2 = atm_meta.get("r_squared")
    if atm_meta["correction_applied"]:
        print(f"  {d1} -> {d2}: atmospheric correction applied (R2={r2:.2f})")
    else:
        print(f"  {d1} -> {d2}: atmospheric correction skipped (R2={r2!r})")

16:05:02 INFO [  atmosphere] Elevation correlation too weak (R²=0.00, best candidate slope=-0.05501 rad/m) — skipping atmospheric correction to avoid absorbing real deformation signal.
  2016-07-21 -> 2016-08-02: atmospheric correction skipped (R2=0.0005533695220947266)
16:05:06 INFO [  atmosphere] Elevation correlation too weak (R²=0.00, best candidate slope=-0.05631 rad/m) — skipping atmospheric correction to avoid absorbing real deformation signal.
  2016-07-21 -> 2016-08-14: atmospheric correction skipped (R2=0.0004646182060241699)
16:05:10 INFO [  atmosphere] Elevation correlation too weak (R²=0.00, best candidate slope=-0.04203 rad/m) — skipping atmospheric correction to avoid absorbing real deformation signal.
  2016-07-21 -> 2016-08-26: atmospheric correction skipped (R2=0.0006348490715026855)
16:05:13 INFO [  atmosphere] Elevation correlation too weak (R²=0.00, best candidate slope=-0.05250 rad/m) — skipping atmospheric correction to avoid absorbing real deformation signal.
  

In [12]:
atm_corrector = AtmosphericCorrector(method="elevation")
corrected_interferograms = {}

for (d1, d2), result in interferograms.items():
    phase = np.angle(result.interferogram)
    corrected, meta = atm_corrector.correct(phase, dem=dem_path,profile=result.profile, return_metadata=True)
    corrected_interferograms[(d1, d2)] = corrected
    print(f"  {d1} -> {d2}: correction_applied={meta['correction_applied']}, R²={meta.get('r_squared')}")

16:06:06 INFO [  atmosphere] Elevation correlation too weak (R²=0.00, best candidate slope=-0.05501 rad/m) — skipping atmospheric correction to avoid absorbing real deformation signal.
  2016-07-21 -> 2016-08-02: correction_applied=False, R²=0.0005533695220947266
16:06:10 INFO [  atmosphere] Elevation correlation too weak (R²=0.00, best candidate slope=-0.05631 rad/m) — skipping atmospheric correction to avoid absorbing real deformation signal.
  2016-07-21 -> 2016-08-14: correction_applied=False, R²=0.0004646182060241699
16:06:13 INFO [  atmosphere] Elevation correlation too weak (R²=0.00, best candidate slope=-0.04203 rad/m) — skipping atmospheric correction to avoid absorbing real deformation signal.
  2016-07-21 -> 2016-08-26: correction_applied=False, R²=0.0006348490715026855
16:06:17 INFO [  atmosphere] Elevation correlation too weak (R²=0.00, best candidate slope=-0.05250 rad/m) — skipping atmospheric correction to avoid absorbing real deformation signal.
  2016-07-21 -> 2016-09

## 8. Phase unwrapping — every real pair

In [13]:
unwrapper = PhaseUnwrapper(cost_mode="defo", init_method="mcf")
UNWRAP_LOOKS_AZ, UNWRAP_LOOKS_RG = 8, 4
TOTAL_LOOKS = LOOKS_AZ * LOOKS_RG * UNWRAP_LOOKS_AZ * UNWRAP_LOOKS_RG

unwrapped_results, conncomp_results, reliability = {}, {}, {}

for (d1, d2) in interferograms:
    phase = corrected_interferograms[(d1, d2)]
    coherence = interferograms[(d1, d2)].coherence

    phase_ml = multilook(phase, UNWRAP_LOOKS_AZ, UNWRAP_LOOKS_RG, wrapped_phase=True)
    coh_ml = multilook(coherence, UNWRAP_LOOKS_AZ, UNWRAP_LOOKS_RG, wrapped_phase=False)

    unwrapped, conncomp = unwrapper.unwrap(
        phase_ml, coh_ml, nlooks=float(TOTAL_LOOKS), min_conncomp_frac=0.001, min_region_size=100,
    )
    unwrapped_results[(d1, d2)] = unwrapped
    conncomp_results[(d1, d2)] = conncomp
    reliability[(d1, d2)] = 100 * np.mean(conncomp > 0)
    print(f"  {d1} -> {d2}: coherence={coh_ml.mean():.3f}, reliable={reliability[(d1,d2)]:5.1f}%")

print(f"\nMean reliable coverage: {np.mean(list(reliability.values())):.1f}%")

16:07:05 INFO [      unwrap] Unwrapping 155x2320 pixels (cost=defo, init=mcf, nlooks=64.0)

snaphu v2.0.7
22 parameters input from file /tmp/tmpm5mnck21/snaphu.config.tbbtmvmu.txt (22 lines total)
Reading wrapped phase from file /tmp/tmpm5mnck21/snaphu.igram.bqazsrj9.c8
No weight file specified.  Assuming uniform weights
Reading correlation data from file /tmp/tmpm5mnck21/snaphu.corr.hnygw8ab.f4
Calculating deformation-mode cost parameters
Initializing flows with MCF algorithm
Running nonlinear network flow optimizer
Maximum flow on network: 3
Flow increment: 1  (Total improvements: 0)
Found 1 valid set(s) of connected nodes
Flow increment: 2  (Total improvements: 3807)
Found 1 valid set(s) of connected nodes
Growing connected component mask
Writing connected components to file /tmp/tmpm5mnck21/snaphu.conncomp.sf64e4sk.u4 as 4-byte unsigned ints
Maximum flow on network: 2
Total solution cost: 101336645
Integrating phase
Writing output to file /tmp/tmpm5mnck21/snaphu.unw.p11tatf2.f4
Pro

## 9. Baseline-optimized network

Uses the real, package-level `perpendicular_baseline()` (verified
against 4 hand-computable geometry cases earlier this project) instead
of a duplicated, inline copy.

In [14]:
from pygeofetch.insar.geolocation import geodetic_to_ecef, find_zero_doppler_time, interpolate_orbit_state

scene_lat, scene_lon = 42.70, 13.29  # real, approximate Amatrice epicentral area
ground_point = geodetic_to_ecef(scene_lat, scene_lon, 0.0)

baselines = []
for d1, d2 in interferograms:
    if d1 not in orbit_files or d2 not in orbit_files:
        continue
    ref_orbit = parse_orbit_file(orbit_files[d1])
    sec_orbit = parse_orbit_file(orbit_files[d2])
    t_ref = find_zero_doppler_time(*ref_orbit, ground_point, datetime.strptime(d1, "%Y-%m-%d"))
    t_sec = find_zero_doppler_time(*sec_orbit, ground_point, datetime.strptime(d2, "%Y-%m-%d"))
    pos_ref, _ = interpolate_orbit_state(*ref_orbit, t_ref)
    pos_sec, _ = interpolate_orbit_state(*sec_orbit, t_sec)
    b_perp = perpendicular_baseline(pos_ref, pos_sec, ground_point)
    baselines.append((d1, d2, b_perp))
    print(f"  {d1} -> {d2}: baseline={b_perp:.1f}m")

baselines_sorted = sorted(baselines, key=lambda x: x[2])
parent = {d: d for d in extracted_dates}
def find(d):
    while parent[d] != d: d = parent[d]
    return d

network_pairs = []
for d1, d2, b in baselines_sorted:
    r1, r2 = find(d1), find(d2)
    if r1 != r2:
        parent[r1] = r2
        network_pairs.append((d1, d2))

connected = {d for pair in network_pairs for d in pair}
print(f"\nReal baseline-optimized network: {len(network_pairs)} pairs, {len(connected)}/{len(extracted_dates)} dates connected")

16:10:28 INFO [ geolocation] Parsed 9361 orbit state vectors from S1A_OPER_AUX_POEORB_OPOD_20210312T234800_V20160720T225943_20160722T005943.EOF
16:10:28 INFO [ geolocation] Parsed 9361 orbit state vectors from S1A_OPER_AUX_POEORB_OPOD_20210313T044206_V20160801T225943_20160803T005943.EOF
  2016-07-21 -> 2016-08-02: baseline=216.1m
16:10:29 INFO [ geolocation] Parsed 9361 orbit state vectors from S1A_OPER_AUX_POEORB_OPOD_20210312T234800_V20160720T225943_20160722T005943.EOF
16:10:29 INFO [ geolocation] Parsed 9361 orbit state vectors from S1A_OPER_AUX_POEORB_OPOD_20210313T093737_V20160813T225943_20160815T005943.EOF
  2016-07-21 -> 2016-08-14: baseline=189.0m
16:10:29 INFO [ geolocation] Parsed 9361 orbit state vectors from S1A_OPER_AUX_POEORB_OPOD_20210312T234800_V20160720T225943_20160722T005943.EOF
16:10:30 INFO [ geolocation] Parsed 9361 orbit state vectors from S1A_OPER_AUX_POEORB_OPOD_20210313T134622_V20160825T225943_20160827T005943.EOF
  2016-07-21 -> 2016-08-26: baseline=268.5m
16:1

## 10. Real, georeferenced reference pixel — stable footwall, away from the fault

In [15]:
REF_LAT, REF_LON = 42.80, 13.15  # real, footwall area away from the published deformation lobes

reference_pair = next(iter(interferograms))
reference_transform = interferograms[reference_pair].profile["transform"]

ref_row_native, ref_col_native = rasterio.transform.rowcol(reference_transform, REF_LON, REF_LAT)
ref_row_ml = ref_row_native // (LOOKS_AZ * UNWRAP_LOOKS_AZ)
ref_col_ml = ref_col_native // (LOOKS_RG * UNWRAP_LOOKS_RG)

min_r = min(u.shape[0] for k, u in unwrapped_results.items() if k in network_pairs) if network_pairs else min(u.shape[0] for u in unwrapped_results.values())
min_c = min(u.shape[1] for k, u in unwrapped_results.items() if k in network_pairs) if network_pairs else min(u.shape[1] for u in unwrapped_results.values())
REF_PIXEL = (min(max(ref_row_ml, 0), min_r - 1), min(max(ref_col_ml, 0), min_c - 1))
print(f"Real, georeferenced reference pixel: {REF_PIXEL}")

Real, georeferenced reference pixel: (np.int32(57), np.int32(922))


## 11. Bridging — exclude unreliable pairs, never corrupt the rest

In [16]:
sbas_pairs, excluded_pairs = [], []

for (d1, d2) in network_pairs:
    unwrapped = unwrapped_results[(d1, d2)]
    conncomp = conncomp_results[(d1, d2)]
    coherence_ml = multilook(interferograms[(d1, d2)].coherence, UNWRAP_LOOKS_AZ, UNWRAP_LOOKS_RG, wrapped_phase=False)

    unwrapped_c = unwrapped[:min_r, :min_c]
    conncomp_c = conncomp[:min_r, :min_c]
    coherence_c = coherence_ml[:min_r, :min_c]

    if conncomp_c[REF_PIXEL] == 0:
        print(f"  {d1} -> {d2}: reference pixel not reliable -- EXCLUDING")
        excluded_pairs.append((d1, d2))
        continue

    bridged, offsets = bridge_unwrap_regions(
        unwrapped_c, conncomp_c, bridge_radius=50, min_region_size=100, reference_pixel=REF_PIXEL,
    )
    sbas_pairs.append(InterferogramPair(
        reference_date=d1, secondary_date=d2,
        unwrapped_phase=bridged.astype(np.float32), coherence=coherence_c.astype(np.float32),
    ))
    print(f"  {d1} -> {d2}: bridged and included")

print(f"\n{len(sbas_pairs)}/{len(network_pairs)} pairs usable; excluded: {excluded_pairs}")
network_check = DataValidator.validate_sbas_network(sbas_pairs, extracted_dates)
print(f"Network valid: {network_check.valid}")

  2016-09-07 -> 2016-09-19: reference pixel not reliable -- EXCLUDING
  2016-08-14 -> 2016-09-07: reference pixel not reliable -- EXCLUDING
  2016-08-02 -> 2016-08-14: reference pixel not reliable -- EXCLUDING
  2016-08-14 -> 2016-08-26: reference pixel not reliable -- EXCLUDING
  2016-07-21 -> 2016-09-19: reference pixel not reliable -- EXCLUDING

0/5 pairs usable; excluded: [('2016-09-07', '2016-09-19'), ('2016-08-14', '2016-09-07'), ('2016-08-02', '2016-08-14'), ('2016-08-14', '2016-08-26'), ('2016-07-21', '2016-09-19')]
Network valid: False


## 12. SBAS if connected, honest single-pair LOS result if not

In [17]:
if network_check.valid and len(extracted_dates) > 2 and len(sbas_pairs) >= 2:
    sbas = SBASTimeSeries(reference_date=sorted(sbas_pairs, key=lambda p: p.reference_date)[0].reference_date, use_gpu=False)
    ts_result = sbas.invert(sbas_pairs, coherence_threshold=0.3, reference_pixel=REF_PIXEL)
    velocity_mm_yr = ts_result.velocity * 1000
    print(f"Real SBAS LOS velocity range: [{np.nanmin(velocity_mm_yr):.1f}, {np.nanmax(velocity_mm_yr):.1f}] mm/year")
    best_displacement_m = None
else:
    print("Reporting the single, real coseismic LOS pair directly.")
    best_pair = max(sbas_pairs, key=lambda p: reliability.get((p.reference_date, p.secondary_date), 0)) if sbas_pairs else None
    if best_pair is not None:
        d1, d2 = best_pair.reference_date, best_pair.secondary_date
        best_displacement_m = best_pair.unwrapped_phase * WAVELENGTH_M / (4 * np.pi)
        print(f"Real coseismic pair: {d1} -> {d2}")
        print(f"Real LOS displacement range: [{np.nanmin(best_displacement_m)*100:.2f}, {np.nanmax(best_displacement_m)*100:.2f}] cm")
    else:
        best_displacement_m = None
        print("No usable pairs survived bridging -- no LOS result to report.")

Reporting the single, real coseismic LOS pair directly.
No usable pairs survived bridging -- no LOS result to report.


## 13. Real map of the final LOS displacement result

In [18]:
if best_displacement_m is not None:
    disp_path = output_dir / "displacement_cm.tif"
    profile = {
        "driver": "GTiff", "count": 1, "height": best_displacement_m.shape[0],
        "width": best_displacement_m.shape[1], "crs": interferograms[reference_pair].profile.get("crs"),
        "transform": reference_transform, "dtype": "float32", "nodata": -9999.0,
    }
    with rasterio.open(disp_path, "w", **profile) as dst:
        dst.write((best_displacement_m * 100).astype(np.float32)[np.newaxis])

    with rasterio.open(disp_path) as src:
        bounds = src.bounds
    center_lat, center_lon = (bounds.bottom + bounds.top) / 2, (bounds.left + bounds.right) / 2

    mv_disp = MapViewer(center=(center_lat, center_lon), zoom=11)
    mv_disp.add_basemap("SATELLITE")
    disp_vmin = float(np.nanpercentile(best_displacement_m, 2) * 100)
    disp_vmax = float(np.nanpercentile(best_displacement_m, 98) * 100)
    mv_disp.add_raster(str(disp_path), colormap="RdBu_r", layer_name=f"LOS displacement cm ({d1} -> {d2})", vmin=disp_vmin, vmax=disp_vmax)
    mv_disp.show()

## 14. Honest summary

This notebook produces a real LOS displacement result using every real
fix from this project. Coherence in the 0.2-0.3 range for this
vegetated, mountainous AOI is expected, not a processing failure —
confirmed against ESA training material and, if run, against the
Mexico City positive-control test built alongside this project. Real,
reliable coverage (the `reliable=` percentage from unwrapping) is the
number that matters for trusting any specific pixel, not coherence
alone.